In [10]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster, GroupedLayerControl
from folium import FeatureGroup  # Correct import


In [11]:
# Load airport data
df = pd.read_csv("https://davidmegginson.github.io/ourairports-data/airports.csv")
df = df[df["iso_country"] == "CA"].copy()  # Only Canadian aerodromes
df.dropna(subset=["ident", "latitude_deg", "longitude_deg"], inplace=True)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude_deg, df.latitude_deg), crs="EPSG:4326")

In [12]:
# Define airport icon styles
ICON_STYLE = {
    "heliport": {"icon": "helicopter", "color": "green"},
    "seaplane_base": {"icon": "ship", "color": "cadetblue"},
    "small_airport": {"icon": "circle", "color": "gray"},
    "medium_airport": {"icon": "plane", "color": "blue"},
    "large_airport": {"icon": "plane", "color": "darkblue"},
    "default": {"icon": "map-marker", "color": "lightgray"},
}

# Create the map
m = folium.Map(location=[44.0, -79.0], zoom_start=8)


In [13]:
# Create feature groups for each province
province_parents = {}
province_names = sorted(gdf["iso_region"].unique())

In [23]:
# Create the feature groups
for prov in province_names:
    fg = FeatureGroup(name=prov).add_to(m)
    province_parents[prov] = fg

    # Group by airport type within the province
    for atype, sub in gdf[gdf['iso_region'] == prov].groupby("type"):
        cluster = MarkerCluster().add_to(fg)
        style = ICON_STYLE.get(atype, ICON_STYLE["default"])
        
        # Add markers to the cluster
        for _, row in sub.iterrows():
            lat, lon = row.geometry.y, row.geometry.x
            folium.Marker(
                [lat, lon],
                tooltip=f"{row['ident']} • {row['name']}",
                popup=f"{row['ident']} – {row['name']}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}",
                icon=folium.Icon(color=style["color"], icon=style["icon"], prefix="fa")
            ).add_to(cluster)

    fg.add_to(m)  #now  add the FeatureGroup to the map after all markers have been added


In [24]:
# Prepare a list of group names and corresponding layers
group_names = list(province_parents.keys())
layers = list(province_parents.values())

# Prepare grouping for the layer control
layer_groups = {prov: province_parents[prov] for prov in province_names}




In [26]:
# Add GroupedLayerControl
GroupedLayerControl(
    groups=groups,  # List of group names (provinces)
    layers=layers,  # Corresponding layers
    collapsed=False
).add_to(m)




NameError: name 'groups' is not defined

In [28]:
layer_groups

{'CA-AB': <folium.map.FeatureGroup at 0x10e45ddc2f0>,
 'CA-BC': <folium.map.FeatureGroup at 0x10e49634050>,
 'CA-MB': <folium.map.FeatureGroup at 0x10e46e62e70>,
 'CA-NB': <folium.map.FeatureGroup at 0x10e46f6ef90>,
 'CA-NL': <folium.map.FeatureGroup at 0x10e46f6ef00>,
 'CA-NS': <folium.map.FeatureGroup at 0x10e4733b170>,
 'CA-NT': <folium.map.FeatureGroup at 0x10e4739ddc0>,
 'CA-NU': <folium.map.FeatureGroup at 0x10e47412210>,
 'CA-ON': <folium.map.FeatureGroup at 0x10e474793d0>,
 'CA-PE': <folium.map.FeatureGroup at 0x10e46b6c590>,
 'CA-QC': <folium.map.FeatureGroup at 0x10e4751b0e0>,
 'CA-SK': <folium.map.FeatureGroup at 0x10e4736db50>,
 'CA-YT': <folium.map.FeatureGroup at 0x10e4773ac30>}